In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import date
from pyspark.sql.functions import *
from pyspark.sql import functions as F
import time

spark = SparkSession.builder.master("local[*]").appName("unoptimized_app").getOrCreate()

# Функция для подсчета затрченного времени на действие
#def measure_time(query):
# start = time.time()
# query.count()
# end = time.time()
# print(f"Затраченное время: {end - start:.2f} sec.")
    
#Подготовка данных
#1.1 Загрузка данных
#создать схему для данных
shema_books = StructType([
  StructField("ISBN",StringType(), True),
  StructField("Book-Title",StringType(), True),
  StructField("Book-Author", StringType(), True),
  StructField("Year-Of-Publication", IntegerType(), True),
  StructField("Publisher", StringType(), True),
  StructField("Image-URL-S", StringType(), True),
  StructField("Image-URL-M", StringType(), True),
  StructField("Image-URL-L", StringType(), True)
 ])

shema_rating = StructType([
  StructField("User-ID",IntegerType(), True),
  StructField("ISBN",StringType(), True), 
  StructField("Book-Rating",IntegerType(), True)
])

shema_users = StructType([
  StructField("User-ID",IntegerType(), True),
  StructField("Location",StringType(), True), 
  StructField("Age",IntegerType(), True)
])

#ArrayType(StringType(), True)
df_books = spark.read.csv("books.csv", 
                          header=True, 
                          schema=shema_books,  
                          sep=";")

df_rating = spark.read.csv("ratings.csv", 
                           header=True, 
                           schema=shema_rating, 
                           sep=";") 

df_users = spark.read.csv("users.csv", 
                          header=True, 
                          schema=shema_users, 
                          sep=";")

#1.2 Очистка данных
new_coll = [c.replace("-","_").lower() for c in df_books.columns]
new_coll_rating = [c.replace("-","_").lower() for c in df_rating.columns]
new_coll_users = [c.replace("-","_").lower() for c in df_users.columns]

df_books = df_books.toDF(*new_coll)
df_rating = df_rating.toDF(*new_coll_rating)
df_users = df_users.toDF(*new_coll_users)

#df_books.show(5)
#df_rating.show(5)
#df_users.show(20,truncate=False)

#df_books.printSchema()
#df_rating.printSchema()
#df_users.printSchema()

#df_books.summary().show()
#df_rating.summary().show()
#df_users.summary().show()

#2.1: Топ-10 книг по среднему рейтингу
start = time.time()
df_rating = df_rating.where(df_rating.book_rating>0)
df_rating_agg= (df_rating.groupBy("isbn").agg(F.count("isbn").alias("rating_count"), F.avg("book_rating").alias("rating_avg")).
                 where(F.col("rating_count")>300))
                
df_books_join = df_books.join(df_rating_agg, df_books.isbn == df_rating_agg.isbn, "inner")
df_books_join = df_books_join.sort(df_books_join.rating_avg.desc()).select("book_title", "book_author", "year_of_publication", "rating_avg")
df_books_join.show(10)
#measure_time(df_books_join)
#df_books_join.explain()

#2.2: Извлечение страны и объединение таблиц
df_users = df_users.withColumn("contry", element_at(F.split(F.col("location"), "[,]"),-1)) 
df_rating_join = df_rating.join(df_users, df_rating.user_id==df_users.user_id, "inner").select(df_rating.user_id,df_rating.book_rating,df_users.contry) 
#df_rating_join.show()
#measure_time(df_rating_join)
#df_rating_join.explain()

#2.3: Топ-5 стран по среднему рейтингу
df_rating_join = (df_rating_join.groupBy("contry").
                   agg(F.count("user_id").alias("rating_count"), F.avg("book_rating").alias("avg_country_rating")).
                   orderBy(F.col("rating_count").desc()))
df_rating_join.show(5)
#measure_time(df_rating_join)
#df_rating_join.explain()

end = time.time()
print(f"Затраченное время без оптимизации: {end - start:.2f} sec.")

#Block optimized
start = time.time()
df_rating = df_rating.where(df_rating.book_rating>0)
df_rating.cache()
df_rating_agg2=(df_rating.groupBy("isbn").agg(F.count("isbn").alias("rating_count"), F.avg("book_rating").alias("rating_avg")).
                 where(F.col("rating_count")>300)) 
#броадкаст 
df_books_join2 = df_books.join(broadcast(df_rating_agg2), df_books.isbn == df_rating_agg2.isbn, "inner")
df_books_join2 = df_books_join2.sort(df_books_join2.rating_avg.desc()).select("book_title", "book_author", "year_of_publication", "rating_avg")
df_books_join2.show(10)
#measure_time(df_books_join)
#df_books_join.explain()

#2.2: Извлечение страны и объединение таблиц
df_users = df_users.withColumn("contry", element_at(F.split(F.col("location"), "[,]"),-1)) 
df_rating_join2 = df_rating.join(df_users, df_rating.user_id==df_users.user_id, "inner").select(df_rating.user_id,df_rating.book_rating,df_users.contry) 
#df_rating.unpersist()
#df_rating_join.cache()
#df_rating_join.show()
#measure_time(df_rating_join)
#df_rating_join.explain()

#2.3: Топ-5 стран по среднему рейтингу
df_rating_join2 =  (df_rating_join2.groupBy("contry").
                   agg(F.count("user_id").alias("rating_count"), F.avg("book_rating").alias("avg_country_rating")).
                   orderBy(F.col("rating_count").desc()))

df_rating_join2.show(5)
#measure_time(df_rating_join)
#df_rating_join.explain()

end = time.time()
print(f"Затраченное время с оптимизацией: {end - start:.2f} sec.")
spark.stop()


+--------------------+-------------+-------------------+------------------+
|          book_title|  book_author|year_of_publication|        rating_avg|
+--------------------+-------------+-------------------+------------------+
|Harry Potter and ...|J. K. Rowling|               1999| 8.939297124600639|
|The Secret Life o...|Sue Monk Kidd|               2003| 8.452768729641694|
|   The Da Vinci Code|    Dan Brown|               2003| 8.435318275154003|
|The Lovely Bones:...| Alice Sebold|               2002| 8.185289957567186|
|The Red Tent (Bes...|Anita Diamant|               1998| 8.182767624020888|
|Divine Secrets of...|Rebecca Wells|               1997|            7.8875|
|         Wild Animus| Rich Shapero|               2004|4.3907056798623065|
+--------------------+-------------+-------------------+------------------+

+---------------+------------+------------------+
|         contry|rating_count|avg_country_rating|
+---------------+------------+------------------+
|            